In [2]:
import pandas as pd

In [3]:
adm = pd.read_csv(r'C:\Users\sathv\Downloads\mimic-iv-clinical-database-demo-2.2\mimic-iv-clinical-database-demo-2.2\hosp\admissions.csv.gz')

diag_icd = pd.read_csv(r'C:\Users\sathv\OneDrive\Desktop\MIMIC DEMO\mimic-iv-clinical-database-demo-2.2\mimic-iv-clinical-database-demo-2.2\hosp\diagnoses_icd.csv.gz')
proced_icd = pd.read_csv(r'C:\Users\sathv\OneDrive\Desktop\MIMIC DEMO\mimic-iv-clinical-database-demo-2.2\mimic-iv-clinical-database-demo-2.2\hosp\procedures_icd.csv.gz')
d_icd_diag = pd.read_csv(r'C:\Users\sathv\OneDrive\Desktop\MIMIC DEMO\mimic-iv-clinical-database-demo-2.2\mimic-iv-clinical-database-demo-2.2\hosp\d_icd_diagnoses.csv.gz')
d_icd_proced = pd.read_csv(r'C:\Users\sathv\OneDrive\Desktop\MIMIC DEMO\mimic-iv-clinical-database-demo-2.2\mimic-iv-clinical-database-demo-2.2\hosp\d_icd_procedures.csv.gz')

In [4]:
diag_full = diag_icd.merge(
    d_icd_diag,
    on=['icd_code', 'icd_version'],
    how='left'
)

In [5]:
diag_full.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version,long_title
0,10035185,22580999,3,4139,9,Other and unspecified angina pectoris
1,10035185,22580999,10,V707,9,Examination of participant in clinical trial
2,10035185,22580999,1,41401,9,Coronary atherosclerosis of native coronary ar...
3,10035185,22580999,9,3899,9,Unspecified hearing loss
4,10035185,22580999,11,V8532,9,"Body Mass Index 32.0-32.9, adult"


In [6]:
proced_full = proced_icd.merge(
    d_icd_proced,
    on=['icd_code','icd_version'],
    how = 'left'
)

In [7]:
proced_full.head()

,subject_id,hadm_id,seq_num,chartdate,icd_code,icd_version,long_title
0,10011398,27505812,3,2146-12-15,3961,9,Extracorporeal circulation auxiliary to open h...
1,10011398,27505812,2,2146-12-15,3615,9,Single internal mammary-coronary artery bypass
2,10011398,27505812,1,2146-12-15,3614,9,(Aorto)coronary bypass of four or more coronar...
3,10014729,23300884,4,2125-03-23,3897,9,Central venous catheter placement with guidance
4,10014729,23300884,1,2125-03-20,3403,9,Reopening of recent thoracotomy site


PATIENT SUMMARY


In [8]:
def patient_summary(subject_id,hadm_id):
    adm_info = adm[
        (adm['subject_id'] == subject_id) &
        (adm['hadm_id'] == hadm_id)
    ]
    
    patient_diag = diag_full[
        (diag_icd['subject_id'] == subject_id) &
        (diag_icd['hadm_id'] == hadm_id)
    ]
    
    patient_proc = proced_full[
        (proced_icd['subject_id'] == subject_id) &
        (proced_icd['hadm_id'] == hadm_id)
    ]
    
    print("ADMISSION INFO")
    print(adm_info[['subject_id','hadm_id','admittime','dischtime','admission_type','edregtime','edouttime','hospital_expire_flag']])
    
    print("\nDIAGNOSES")
    print(patient_diag[['icd_code','long_title']])
    
    print("\nPROCEDURES")
    print(
        patient_proc[
            ['icd_code', 'long_title']
        ])

In [9]:
patient_summary(10011398,27505812)

ADMISSION INFO
     subject_id   hadm_id            admittime            dischtime  \
257    10011398  27505812  2146-12-15 07:15:00  2146-12-19 13:37:00   

                  admission_type edregtime edouttime  hospital_expire_flag  
257  SURGICAL SAME DAY ADMISSION       NaN       NaN                     0  

DIAGNOSES
    icd_code                                         long_title
310     5641                           Irritable bowel syndrome
311    60000  Hypertrophy (benign) of prostate without urina...
312    V1301                Personal history of urinary calculi
313     4779               Allergic rhinitis, cause unspecified
314     2724               Other and unspecified hyperlipidemia
315    V1272                 Personal history of colonic polyps
316     4239                 Unspecified disease of pericardium
317     4111                     Intermediate coronary syndrome
318    41401  Coronary atherosclerosis of native coronary ar...
319     4019                 Unspecif

DISEASES THAT LEAD TO HOSPITAL MORTALITY

In [10]:
diag_full.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version,long_title
0,10035185,22580999,3,4139,9,Other and unspecified angina pectoris
1,10035185,22580999,10,V707,9,Examination of participant in clinical trial
2,10035185,22580999,1,41401,9,Coronary atherosclerosis of native coronary ar...
3,10035185,22580999,9,3899,9,Unspecified hearing loss
4,10035185,22580999,11,V8532,9,"Body Mass Index 32.0-32.9, adult"


In [11]:
diag_mort = diag_full.merge(
    adm[['subject_id','hadm_id','hospital_expire_flag']],
    on=['subject_id','hadm_id'],
    how = 'left'
)

In [12]:
diag_mort.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version,long_title,hospital_expire_flag
0,10035185,22580999,3,4139,9,Other and unspecified angina pectoris,0
1,10035185,22580999,10,V707,9,Examination of participant in clinical trial,0
2,10035185,22580999,1,41401,9,Coronary atherosclerosis of native coronary ar...,0
3,10035185,22580999,9,3899,9,Unspecified hearing loss,0
4,10035185,22580999,11,V8532,9,"Body Mass Index 32.0-32.9, adult",0


In [13]:
mortality_by_disease = (
    diag_mort.groupby('long_title')['hospital_expire_flag']
    .mean()
    .sort_values(ascending=False)
)

In [14]:
mortality_by_disease.head()

long_title
Chronic combined systolic (congestive) and diastolic (congestive) heart failure            1.0
Epilepsy, unspecified, not intractable, with status epilepticus                            1.0
Personal history of other malignant neoplasm of rectum, rectosigmoid junction, and anus    1.0
Dependence on supplemental oxygen                                                          1.0
Ileus, unspecified                                                                         1.0
Name: hospital_expire_flag, dtype: float64

In [15]:
adm['hospital_expire_flag'].value_counts()

hospital_expire_flag
0    260
1     15
Name: count, dtype: int64

In [18]:
disease_counts = diag_mort['long_title'].value_counts(ascending=False)

In [19]:
common_diseases = disease_counts[disease_counts >= 50].index

mortality_by_disease = (
    diag_mort[diag_mort['long_title'].isin(common_diseases)]
    .groupby('long_title')['hospital_expire_flag']
    .mean()
    .sort_values(ascending=False)
)

In [20]:
mortality_by_disease.head(10)

long_title
Acute kidney failure, unspecified       0.125000
Other and unspecified hyperlipidemia    0.072727
Hyperlipidemia, unspecified             0.052632
Unspecified essential hypertension      0.044118
Name: hospital_expire_flag, dtype: float64